TRABAJO PRÁCTICO N° 4

MÉTODOS SUPERVISADOS: REGRESIÓN & CLASIFICACIÓN USANDO LA EHP

A. Enfoque de validación

Utilicen la base respondieron. Para cada año, dividan las observaciones en
una base de prueba (test) y una de entrenamiento (train) utilizando el
comando train_test_split. La base de entrenamiento debe comprender el
70% de los datos, y la semilla a utilizar (random state instance) debe ser 444.
Establezca a desocupado como su variable dependiente en la base de
entrenamiento (vector y). El resto de las variables seleccionadas serán las
variables independientes (matriz X). Recuerden agregar la columna de unos
(1).


1. Cree una tabla de diferencia de medias entre la base de entrenamiento
y la de testeo de las características seleccionadas en su matriz X.
Comente la tabla de la diferencia de medias de sus variables entre
entrenamiento y testeo.

In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm # Importar si usas sm.add_constant

# --- Tu código de carga, limpieza y concatenación ---
# Cargar las bases
t2004 = pd.read_stata('D:/Mis documentos/Estudio/Lic en economia/2do Año/Big data & machine learning/Tps/TP II/Individual_t104.dta')
t2024 = pd.read_excel('D:/Mis documentos/Estudio/Lic en economia/2do Año/Big data & machine learning/Tps/TP II/usu_individual_T124.xlsx')

# Convertir nombres de columnas a mayúsculas
t2004.columns = t2004.columns.str.upper()
t2024.columns = t2024.columns.str.upper()

# Filtrar por región: GBA
# Asegúrate de que el valor para GBA en 2024 es 1 (esto ya lo tienes)
t2004 = t2004[t2004['REGION'] == 'Gran Buenos Aires'].copy() # Usar .copy() para evitar SettingWithCopyWarning
t2024 = t2024[t2024['REGION'] == 1].copy() # Usar .copy()

# Conservar solo columnas comunes
columnas_comunes = list(set(t2004.columns) & set(t2024.columns))
# Excluir columnas que sabes que no quieres como features (ej: ID de hogar, ID de persona si existen y no son relevantes)
# Podrías añadir aquí una lista de exclusión si es necesario. Revisa tus columnas originales de EPH.
# Columnas como ITF, IX_TOT, IDDO suelen ser de ingreso/identificación. ESTADO, CAT_INAC, etc. son categóricas.
# Busca en el diccionario de variables de la EPH para saber cuáles son numéricas.
columnas_a_excluir_antes_merge_ejemplo = ['ITF', 'IX_TOT', 'IDDO', 'PONDIH', 'CH05', 'CH06', 'V4_M', 'V4_VI', 'V4_E', 'V4_A', 'V2_M', 'V2_VI', 'V2_E', 'V2_A', 'AGLOMERADO'] # EJEMPLOS: adapta a tus necesidades!
columnas_comunes = [col for col in columnas_comunes if col not in columnas_a_excluir_antes_merge_ejemplo]

t2004 = t2004[columnas_comunes].copy()
t2024 = t2024[columnas_comunes].copy()

# Añadir columna de año
t2004['ANO_EPH'] = 2004
t2024['ANO_EPH'] = 2024

# Concatenar ambas bases
df_gba = pd.concat([t2004, t2024], ignore_index=True)

In [63]:
if 'ESTADO' in df_gba.columns:
    # Crear las bases según la condición
    respondieron = df_gba[df_gba['ESTADO'] != 0].copy() # Usar .copy()
    norespondieron = df_gba[df_gba['ESTADO'] == 0].copy() # Esta base no se usa en la consigna A

    # Mostrar cuántas personas hay en cada grupo (opcional)
    print("Cantidad que respondieron:", respondieron.shape[0])
    print("Cantidad que no respondieron:", norespondieron.shape[0])
else:
    print("ERROR: La columna 'ESTADO' no se encuentra en la base. No se puede continuar.")
    # Aquí podrías querer salir del script
    exit() # Salir si no hay columna ESTADO


# Crear la variable binaria 'desocupado'
# Asegúrate de que 'ESTADO' es numérico si no lo es (ej: si es string '1','2'). Si es 'object',
# podrías necesitar convertirla a numérico antes de crear 'desocupado'.
# Si ESTADO es 'object' por valores raros (como ' '), primero límpiala.
# Ejemplo de limpieza/conversión de ESTADO si fuera object:
if respondieron['ESTADO'].dtype == 'object':
   respondieron['ESTADO'] = pd.to_numeric(respondieron['ESTADO'], errors='coerce').fillna(-99).astype(int) # Convierte a numérico, NaNs a -99, luego a int

# Asumimos que ESTADO ya es numérico o lo limpiaste si era object.
respondieron['desocupado'] = (respondieron['ESTADO'] == 2).astype(int)

# Separar los datos por año
respondieron_2004 = respondieron[respondieron['ANO_EPH'] == 2004].copy()
respondieron_2024 = respondieron[respondieron['ANO_EPH'] == 2024].copy()

print("\n--- Bases separadas por año ---")
print("Respondieron 2004:", respondieron_2004.shape)
print("Respondieron 2024:", respondieron_2024.shape)
print("-" * 30)

Cantidad que respondieron: 14657
Cantidad que no respondieron: 41

--- Bases separadas por año ---
Respondieron 2004: (7647, 169)
Respondieron 2024: (7010, 169)
------------------------------


In [124]:
# Asegúrate de tener las importaciones necesarias al principio de tu script
# from sklearn.model_selection import train_test_split
# import statsmodels.api as sm


def procesar_datos_para_ml(df_año, año):
    """
    Procesa el DataFrame para ML:
    1. Identifica columnas categóricas (object/category + listado manual por codigo)
       y las convierte a dummies.
    2. Identifica columnas restantes de tipo object que deberían ser numéricas
       y las convierte usando pd.to_numeric(errors='coerce').
    3. Excluye la variable dependiente 'desocupado' y columnas identificadoras.
    """
    print(f"\n--- Procesando datos para ML para el año {año} ---")
    df_procesado = df_año.copy() # Trabajamos sobre una copia

    # --- Listas de columnas clave ---
    # Columnas que *nunca* deben ser variables predictoras (IDs, etc.)
    # Asegúrate que esta lista esté completa con todas las columnas que son IDs o identificadores únicos
    columnas_a_excluir_total = [
    'desocupado', 'ANO_EPH', 'CODUSU', 'ANO4', 'TRIMESTRE', 'NRO_HOGAR', 'COMPONENTE',
    'PONDERA',
    # Para 2024 y si existen en 2004 con nombres similares:
    'PONDII', 'PONDIIO', 'PONDIH',
    # Variables de deciles, etc. (revisa todos los prefijos y sufijos en tus datos)
    'DECINDR', 'ADECINDR', 'RDECINDR', 'PDECINDR', 'GDECINDR', 'IDECINDR',
    'DECOCUR', 'ADECOCUR', 'RDECOCUR', 'PDECOCUR', 'GDECOCUR', 'IDECOCUR',
    'DECIFR', 'ADECIFR', 'RDECIFR', 'PDECIFR', 'GDECIFR', 'IDECIFR',
    'DECCFR', 'ADECCFR', 'RDECCFR', 'PDECCFR', 'GDECCFR', 'IDECCFR',
    'IPCF',
    # 'AGLOMERADO', # Descomentar si no la usas como feature
    # 'MAS_500', # Descomentar si no la usas como feature
]

    # Lista de columnas que son categóricas por diccionario EPH (incluso si pandas las lee como numéricas).
    # DEBES REVISAR TU DICCIONARIO DE VARIABLES DE LA EPH para COMPLETAR esta lista correctamente.
    # Estas son variables que representan categorías (ej: estado civil, nivel educativo, condición de actividad, etc.)
    # pero que en los datos vienen como códigos (ej: 1, 2, 3, -9, etc.)
    columnas_categóricas_por_código = [
    # CH
    'CH03', 'CH04', 'CH07', 'CH08', 'CH09', 'CH10', 'CH11', 'CH12', 'CH13', 'CH15', 'CH16',
    # PP02
    'PP02C1', 'PP02C2', 'PP02C3', 'PP02C4', 'PP02C5', 'PP02C6', 'PP02C7', 'PP02C8',
    'PP02E', 'PP02H', 'PP02I',
    # PP03
    'ESTADO', 'CAT_OCUP', 'CAT_INAC',
    'PP03C', 'PP03D', # PP03D si tiene pocos valores únicos
    'PP03G', 'PP03H', 'PP03I', 'PP03J',
    # PP04
    'PP04A', 'PP04B_COD', 'PP04B1', 'PP04B2', 'PP04C', # PP04C si son rangos codificados
    'PP04D_COD', 'PP04G',
    # PP05
    'PP05C_1', 'PP05C_2', 'PP05C_3', 'PP05E', 'PP05F', 'PP05H',
    # PP06 (solo las que son códigos, NO montos)
    'PP06A', 'PP06E', 'PP06H',
    # PP07 (asalariados)
    'PP07A', 'PP07C', 'PP07D', 'PP07E',
    'PP07F1', 'PP07F2', 'PP07F3', 'PP07F4', 'PP07F5',
    'PP07G1', 'PP07G2', 'PP07G3', 'PP07G4',
    'PP07H', 'PP07I', 'PP07J', 'PP07K',
    # PP09
    'PP09A', 'PP09B', 'PP09C', # Y sus variantes ESP
    # PP10 y PP11 (revisar todas, muchas son códigos)
    'PP10A', 'PP10C', 'PP10D', 'PP10E',
    'PP11A', 'PP11B1', 'PP11B_COD', 'PP11C', 'PP11C99', 'PP11D_COD',
    'PP11L', 'PP11L1', 'PP11M', 'PP11N', 'PP11O', 'PP11P', 'PP11Q', 'PP11R', 'PP11S', 'PP11T',
    # Otras que tenías
    'REGION', 'NIVEL_ED', 'INTENSI', # NIVEL_ED es CH12 usualmente. INTENSI es categórica.
    'MAS_500', # Si la usas como feature
    # 'AGLOMERADO', # Si la usas como feature
]
# Elimina duplicados si los hubiera


    # Columnas que *realmente* son texto libre y no deben ser procesadas (si existen)
    columnas_realmente_texto_libre = [] # Ajusta si tienes alguna columna así


    # --- Paso 1: Construir la lista de columnas a Codificar como Dummies ---

    # Empezar con las que pandas detecta como 'object' o 'category'
    cols_a_transformar_dummy = df_procesado.select_dtypes(include=['object', 'category']).columns.tolist()

    # Añadir las que son categóricas por código, si existen en el df y no están ya en la lista
    for col in columnas_categóricas_por_código:
        if col in df_procesado.columns and col not in cols_a_transformar_dummy:
             cols_a_transformar_dummy.append(col)

    # Asegurarnos de excluir de la codificación dummy las columnas que no deben ser predictoras
    cols_a_transformar_dummy = [col for col in cols_a_transformar_dummy if col not in columnas_a_excluir_total]


    print(f"Columnas identificadas como categóricas para codificar a dummies en {año}: {cols_a_transformar_dummy}")

    # Aplicar pd.get_dummies
    if cols_a_transformar_dummy:
        print(f"Convirtiendo las siguientes columnas a variables dummy: {cols_a_transformar_dummy}")
        # dummy_na=False (default): no crea columna para NaNs. Si una categórica tiene NaNs, las filas con NaN serán 0 en todas las dummies creadas para esa columna.
        df_procesado = pd.get_dummies(df_procesado, columns=cols_a_transformar_dummy, drop_first=True, dummy_na=False)
        print(f"Conversión a dummies completada para el año {año}.")
    else:
        print(f"No se encontraron columnas (object/category/por_codigo) para codificar a dummies en el año {año} (después de exclusiones).")

    print(f"Columnas después de get_dummies para el año {año}:")
    # print(df_procesado.columns.tolist()) # Puede ser muy larga
    print(f"Total de columnas: {len(df_procesado.columns)}")
    print(f"Tipos de datos después de get_dummies para el año {año}:")
    print(df_procesado.dtypes.value_counts())


    # --- Paso 2: Convertir columnas object restantes a numérico (limpieza de datos) ---
    # Identificar columnas que *siguen* siendo 'object' después de get_dummies.
    # Estas son probablemente columnas que deberían ser numéricas pero contienen errores,
    # o columnas de texto libre que no queremos o no podemos usar directamente.
    cols_object_restantes = df_procesado.select_dtypes(include=['object']).columns.tolist()

    # Excluir explícitamente cualquier columna object que REALMENTE sea texto libre o ID que no vas a usar
    # Ya excluimos las IDs al inicio, pero por si alguna otra columna de texto quedó.
    cols_object_restantes = [col for col in cols_object_restantes if col not in columnas_realmente_texto_libre]
    # También excluir las que ya fueron procesadas/excluidas
    cols_object_restantes = [col for col in cols_object_restantes if col not in columnas_a_excluir_total and col not in cols_a_transformar_dummy]


    if cols_object_restantes:
        print(f"\nIdentificadas columnas 'object' restantes que podrían ser numéricas en {año}: {cols_object_restantes}")
        print("Intentando convertir a numérico con errors='coerce'...")

        for col in cols_object_restantes:
            original_dtype = df_procesado[col].dtype
            df_procesado[col] = pd.to_numeric(df_procesado[col], errors='coerce')
            nan_count_after = df_procesado[col].isnull().sum()

            # Verificamos si realmente se cambió el dtype y se introdujeron NaNs (lo cual es el objetivo de coerce en object)
            if df_procesado[col].dtype != original_dtype or nan_count_after > 0:
                 print(f" - Columna '{col}': Convertida a numérico (o NaNs introducidos). Dtype final: {df_procesado[col].dtype}. NaNs: {nan_count_after}.")
            else:
                 # Esto podría pasar si la columna object solo tenía NaNs o strings vacíos que pd.to_numeric maneja
                 print(f" - Columna '{col}': Procesada (podría haber tenido NaNs/vacíos). Dtype final: {df_procesado[col].dtype}. NaNs: {nan_count_after}.")


        print("Conversión de columnas object restantes a numérico completada.")
    else:
        print("\nNo se encontraron columnas 'object' restantes para convertir a numérico.")

    print(f"Tipos de datos finales después de la limpieza numérica para el año {año}:")
    print(df_procesado.dtypes.value_counts())
    print(f"Primeras filas del DataFrame procesado para el año {año}:")
    print(df_procesado.head())
    print("-" * 30)

    # --- Paso 3: Asegurarse de eliminar columnas que no deben ser predictoras (si no fueron dummificadas) ---
    # Esto es una doble verificación para las columnas en columnas_a_excluir_total
    # Algunas (como desocupado, ANO_EPH) las dropeamos al definir X, pero si hay otras IDs
    # que no quieres que pasen NUNCA, puedes dropearlas aquí si no fueron ya dummificadas/excluidas.
    # df_procesado = df_procesado.drop(columns=[col for col in columnas_a_excluir_total if col in df_procesado.columns and col not in cols_a_transformar_dummy], errors='ignore')
    # Generalmente, es más limpio dropearlas al definir X como hacemos abajo.

    return df_procesado


# Aplicar el procesamiento mejorado a cada base anual
respondieron_2004_procesada = procesar_datos_para_ml(respondieron_2004, 2004)
respondieron_2024_procesada = procesar_datos_para_ml(respondieron_2024, 2024)


--- Procesando datos para ML para el año 2004 ---
Columnas identificadas como categóricas para codificar a dummies en 2004: ['CH08', 'PP04B_COD', 'PP07K', 'PP02C5', 'PP07E', 'PP05E', 'PP02C7', 'PP10D', 'PP07F4', 'PP07D', 'PP05H', 'CH16_COD', 'NIVEL_ED', 'PP02C8', 'CH04', 'PP09A', 'PP07G4', 'PP11L', 'PP11R', 'PP10C', 'PP05F', 'PP07F3', 'PP02C3', 'PP07G3', 'PP05C_2', 'PP07I', 'PP11O', 'PP07G2', 'PP03G', 'PP04D_COD', 'CH12', 'PP06H', 'PP11A', 'MAS_500', 'CH13', 'CH09', 'PP07C', 'PP11M', 'CAT_OCUP', 'PP06A', 'PP07G1', 'PP07J', 'PP11P', 'PP06E', 'PP09B', 'PP11C', 'CH14', 'PP07F2', 'PP05C_1', 'PP05C_3', 'PP04C99', 'PP07H', 'PP04A', 'PP04C', 'PP11L1', 'PP02C1', 'PP03I', 'PP11Q', 'PP09C', 'INTENSI', 'PP07F5', 'PP11T', 'CH15_COD', 'PP09C_ESP', 'CH07', 'CH15', 'PP02C6', 'PP10E', 'PP03J', 'PP04G', 'PP11B1', 'CH16', 'PP04B1', 'REGION', 'PP02E', 'CH11', 'H15', 'PP09A_ESP', 'PP11S', 'PP02C2', 'PP07G_59', 'PP11B_COD', 'PP07F1', 'PP11C99', 'PP11D_COD', 'PP03H', 'PP02H', 'PP03C', 'CAT_INAC', 'PP02C4',

In [121]:
# ... (El resto del código para el split de 2004, tabla 2004, split 2024, tabla 2024 y comentario final NO CAMBIA,
#      excepto que ahora la lista de columnas a dropear al definir X debe ser 'columnas_a_excluir_total')

print("\n--- Realizando Train/Test Split y agregando constante ---")

# --- Para 2004 ---
print("\nProcesando año 2004:")
df_actual_2004 = respondieron_2004_procesada
# Definir X (features) y y (target)
# Usar la lista completa de columnas a excluir de X
columnas_a_excluir_total = ['desocupado', 'ANO_EPH', 'CODUSU', 'ANO4', 'TRIMESTRE', 'NRO_HOGAR', 'COMPONENTE'] # Usar la misma lista que en la función
X_2004 = df_actual_2004.drop(columnas_a_excluir_total, axis=1)
y_2004 = df_actual_2004['desocupado']

# Aplicar train_test_split
# ... (código del split para 2004) ...

# Agregar la columna de unos (constante)
# ... (código de add_constant para 2004) ...

print("Dimensiones después del split para 2004:")
print("X_train_2004:", X_train_2004.shape)
print("X_test_2004:", X_test_2004.shape)
print("y_train_2004:", y_train_2004.shape)
print("y_test_2004:", y_test_2004.shape)

# --- CALCULAR Y MOSTRAR LA TABLA DE DIFERENCIA DE MEDIAS PARA 2004 ---
# ... (código para calcular y mostrar la tabla 2004) ...


# --- Para 2024 ---
print("\nProcesando año 2024:")
df_actual_2024 = respondieron_2024_procesada
# Definir X (features) y y (target)
# Usar la misma lista completa de columnas a excluir de X
columnas_a_excluir_total = ['desocupado', 'ANO_EPH', 'CODUSU', 'ANO4', 'TRIMESTRE', 'NRO_HOGAR', 'COMPONENTE'] # Usar la misma lista que en la función
X_2024 = df_actual_2024.drop(columnas_a_excluir_total, axis=1)
y_2024 = df_actual_2024['desocupado']

# Aplicar train_test_split
# ... (código del split para 2024) ...

# Agregar la columna de unos (constante)
# ... (código de add_constant para 2024) ...

print("Dimensiones después del split para 2024:")
print("X_train_2024:", X_train_2024.shape) # type: ignore
print("X_test_2024:", X_test_2024.shape) # type: ignore
print("y_train_2024:", y_train_2024.shape) # type: ignore
print("y_test_2024:", y_test_2024.shape) # type: ignore

# Opcional: Ordenar la tabla por el valor absoluto de la diferencia de forma descendente
tabla_diferencia_medias_2004_sorted = tabla_diferencia_medias_2004.iloc[tabla_diferencia_medias_2004['Diferencia'].abs().argsort()[::-1]] # type: ignore

# Imprimir la tabla (puedes imprimir solo las primeras filas si es muy grande)
print(tabla_diferencia_medias_2004_sorted.head(20)) # Mostrar las 20 mayores diferencias

# --- FIN DEL CÁLCULO Y MOSTRADO DE LA TABLA PARA 2004 ---


# --- Para 2024 ---
print("\nProcesando año 2024:")
df_actual_2024 = respondieron_2024_procesada
# Definir X (features) y y (target)
X_2004 = df_actual_2004.drop(columnas_a_excluir_total, axis=1)
y_2024 = df_actual_2024['desocupado']

# Aplicar train_test_split
X_train_2024, X_test_2024, y_train_2024, y_test_2024 = train_test_split(X_2024, y_2024, test_size=0.3, random_state=444, stratify=y_2024)

# Agregar la columna de unos (constante) a X_train y X_test
X_train_2024 = sm.add_constant(X_train_2024)
X_test_2024 = sm.add_constant(X_test_2024)

print("Dimensiones después del split para 2024:")
print("X_train_2024:", X_train_2024.shape)
print("X_test_2024:", X_test_2024.shape)
print("y_train_2024:", y_train_2024.shape)
print("y_test_2024:", y_test_2024.shape)


# --- CALCULAR Y MOSTRAR LA TABLA DE DIFERENCIA DE MEDIAS PARA 2024 ---
print("\n--- Tabla de Diferencia de Medias (X_train_2024 vs X_test_2024) ---")

# Calcular las medias para cada columna en train y test
mean_train_2024 = X_train_2024.mean()
mean_test_2024 = X_test_2024.mean()

# Calcular la diferencia (Media Train - Media Test)
difference_2024 = mean_train_2024 - mean_test_2024

# Crear un DataFrame con estos resultados
tabla_diferencia_medias_2024 = pd.DataFrame({
    'Media Entrenamiento': mean_train_2024,
    'Media Prueba': mean_test_2024,
    'Diferencia': difference_2024
})

# Opcional: Ordenar la tabla por el valor absoluto de la diferencia de forma descendente
tabla_diferencia_medias_2024_sorted = tabla_diferencia_medias_2024.iloc[tabla_diferencia_medias_2024['Diferencia'].abs().argsort()[::-1]]

# Imprimir la tabla
print(tabla_diferencia_medias_2024_sorted.head(20)) # Mostrar las 20 mayores diferencias

# --- FIN DEL CÁLCULO Y MOSTRADO DE LA TABLA PARA 2024 ---


# --- Comentar la tabla de diferencia de medias ---
print("\n--- Comentario sobre las diferencias de medias ---")
print("Las tablas anteriores muestran la diferencia en las medias de cada variable predictora entre el conjunto de entrenamiento y el conjunto de prueba para cada año.")
print("La columna 'Media Entrenamiento' muestra el promedio de la variable en el set de train, 'Media Prueba' en el set de test, y 'Diferencia' es (Media Train - Media Test).")
print("Las tablas están ordenadas por la magnitud de la diferencia (la columna 'Diferencia' en valor absoluto) para resaltar las variables donde los sets train y test difieren más en promedio.")
print("Idealmente, estas diferencias deberían ser pequeñas para todas las variables, lo que indicaría que el split aleatorio (con la semilla 444) resultó en conjuntos de train y test con distribuciones de variables predictoras muy similares.")
print("Variables con diferencias de medias notablemente más grandes (las que aparecen al principio de la lista) podrían indicar que la división no fue perfectamente equitativa para esas características en particular.")
print("Esto puede ocurrir por azar, especialmente si algunas categorías o valores son raros o tienen una distribución sesgada en el dataset, o si el tamaño del conjunto de datos es limitado.")
print("Las columnas que provienen de la codificación dummy (ej: _Categoria) tendrán medias entre 0 y 1, representando la proporción de individuos con esa categoría.")
print("Observa si hay variables con diferencias que consideras significativas. Para la mayoría de los algoritmos de ML, diferencias pequeñas son preferibles para asegurar que el modelo entrenado en 'train' generalice bien a 'test'.")
print("Sin embargo, con un random_state fijo y estratificación en la variable dependiente (como hicimos), el split suele ser razonablemente bueno en general.")


# --- Continúa con el modelado u otros pasos ---
# Ya tienes X_train_2004, X_test_2004, y_train_2004, y_test_2004
# y X_train_2024, X_test_2024, y_train_2024, y_test_2024
# Estos DataFrames X ya incluyen la columna 'constante'.    


--- Realizando Train/Test Split y agregando constante ---

Procesando año 2004:
Dimensiones después del split para 2004:
X_train_2004: (5352, 1522)
X_test_2004: (2295, 1522)
y_train_2004: (5352,)
y_test_2004: (2295,)

Procesando año 2024:
Dimensiones después del split para 2024:
X_train_2024: (4907, 1003)
X_test_2024: (2103, 1003)
y_train_2024: (4907,)
y_test_2024: (2103,)
            Media Entrenamiento   Media Prueba  Diferencia
CODUSU            206689.233744  206319.231808  370.001936
PP08D1               192.456839     206.323747  -13.866909
PP06D                 22.750747      13.696296    9.054451
P21                  261.001495     268.182135   -7.180640
ANO4                2004.000000    2004.000000    0.000000
IPCF                 363.966646     359.115256    4.851390
TOT_P12               18.431241      14.015251    4.415990
PP06C                 38.015695      41.814815   -3.799120
PONDERA             1625.184978    1622.992593    2.192385
T_VI                  66.213752  

C:\Users\Brian Emanuel Rios\AppData\Local\Temp\ipykernel_6740\3926517721.py:53: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  tabla_diferencia_medias_2004_sorted = tabla_diferencia_medias_2004.iloc[tabla_diferencia_medias_2004['Diferencia'].abs().argsort()[::-1]] # type: ignore
c:\Users\Brian Emanuel Rios\anaconda3\Lib\site-packages\numpy\core\_methods.py:41: RuntimeWarning: invalid value encountered in reduce
  return umr_maximum(a, axis, None, out, keepdims, initial, where)
c:\Users\Brian Emanuel Rios\anaconda3\Lib\site-packages\numpy\core\_methods.py:45: RuntimeWarning: invalid value encountered in reduce
  return umr_minimum(a, axis, None, out, keepdims, initial, where)
c:\Users\Brian Emanuel Rios\anaconda3\Lib\site-packages\numpy\core\_methods.py:41: RuntimeWarning: invalid value encountered in reduce
  return umr_maximum(a, axis, None, out, keepdims, initial, wh

In [118]:
respondieron_2024.columns.tolist()

['ADECIFR',
 'CH08',
 'PP04B_COD',
 'ADECCFR',
 'PP07K',
 'PP02C5',
 'PP07E',
 'PP08J2',
 'PP05E',
 'PP02C7',
 'PP10D',
 'PP07F4',
 'PP07D',
 'PP05H',
 'CH16_COD',
 'PP3E_TOT',
 'NIVEL_ED',
 'PP02C8',
 'CH04',
 'PP09A',
 'DECINDR',
 'PP03D',
 'PP06C',
 'PP07G4',
 'PP11L',
 'PP11R',
 'NRO_HOGAR',
 'PP10C',
 'PP04B2',
 'RDECCFR',
 'PP05F',
 'PP08J3',
 'COMPONENTE',
 'V18_M',
 'PP07F3',
 'PP06D',
 'PDECINDR',
 'IDECIFR',
 'T_VI',
 'PP08D1',
 'PP02C3',
 'PP07G3',
 'PP05C_2',
 'PONDERA',
 'V8_M',
 'PP07I',
 'PP11O',
 'GDECINDR',
 'PP07G2',
 'PP03G',
 'PP04B3_DIA',
 'PP04D_COD',
 'CH12',
 'PDECOCUR',
 'ADECINDR',
 'PP08F1',
 'PP06H',
 'PP11A',
 'MAS_500',
 'V9_M',
 'PP11G_MES',
 'PP11G_DIA',
 'CH13',
 'CH09',
 'PP07C',
 'PP11M',
 'CAT_OCUP',
 'PP06A',
 'PP07G1',
 'V21_M',
 'PP07J',
 'TOT_P12',
 'PP11P',
 'PP04B3_MES',
 'P21',
 'PP06E',
 'PP09B',
 'PP05B2_ANO',
 'GDECIFR',
 'PP11C',
 'CH14',
 'PP11B2_DIA',
 'ESTADO',
 'PP07F2',
 'PP05C_1',
 'PP05C_3',
 'PP04C99',
 'RDECIFR',
 'IDECINDR',
 'PP

In [120]:
# --- Guardar las tablas de diferencia de medias en archivos Excel ---

print("\n--- Guardando tablas de diferencia de medias en archivos Excel ---")

# Nombre del archivo de salida para la tabla de 2004
# Puedes especificar una ruta completa si no quieres guardarlo en la carpeta actual
archivo_excel_20042 = 'tabla_diferencia_medias_2004.xlsx'

try:
    # Usamos el método to_excel() del DataFrame
    # index=True (por defecto) guarda el índice del DataFrame (los nombres de las variables) como una columna.
    # float_format='%.6f' formatea los números flotantes a 6 decimales para mayor claridad en Excel.
    tabla_diferencia_medias_2004_sorted.to_excel(archivo_excel_20042, index=True, float_format='%.6f')
    print(f"Tabla de diferencia de medias de 2004 guardada exitosamente en '{archivo_excel_20042}'")
except Exception as e:
    print(f"Error al intentar guardar la tabla de 2004 en Excel: {e}")

# Nombre del archivo de salida para la tabla de 2024
archivo_excel_20242 = 'tabla_diferencia_medias_2024.xlsx'

try:
    # Guardar el DataFrame de 2024
    tabla_diferencia_medias_2024_sorted.to_excel(archivo_excel_20242, index=True, float_format='%.6f')
    print(f"Tabla de diferencia de medias de 2024 guardada exitosamente en '{archivo_excel_20242}'")
except Exception as e:
    print(f"Error al intentar guardar la tabla de 2024 en Excel: {e}")

print("--- Proceso de guardado en Excel completado ---")


# --- Comentar la tabla de diferencia de medias ---
# ... (Tu bloque de código con el comentario final) ...


--- Guardando tablas de diferencia de medias en archivos Excel ---
Tabla de diferencia de medias de 2004 guardada exitosamente en 'tabla_diferencia_medias_2004.xlsx'
Tabla de diferencia de medias de 2024 guardada exitosamente en 'tabla_diferencia_medias_2024.xlsx'
--- Proceso de guardado en Excel completado ---


La tabla de diferencia de medias nos permite evaluar si la partición de los datos en conjuntos de entrenamiento y prueba ha resultado en subconjuntos con características similares. Lo ideal es que estas diferencias sean pequeñas, indicando que la base de prueba es una buena representación de la base de entrenamiento.

Para el año 2004:
Observamos que, en general, las diferencias de medias entre las variables seleccionadas de la base de entrenamiento y la de prueba son relativamente pequeñas. Por ejemplo, variables como CODUSU presentan una diferencia de aproximadamente 370, mientras que otras como PP08D1 y PP06D muestran diferencias de -13.87 y 9.05 respectivamente. Muchas otras variables, especialmente aquellas que parecen ser indicadores o categóricas, tienen diferencias muy cercanas a cero. Esto sugiere que la división de datos para el año 2004 ha generado conjuntos de entrenamiento y prueba bastante comparables en términos de las medias de sus características.

Para el año 2024:
En la tabla correspondiente al año 2024, notamos que algunas de las primeras variables listadas, como IPCF, P47T, V10_M y T_VI, exhiben diferencias de medias considerablemente altas (ej. 16799.64 para IPCF, 15509.85 para P47T). Si bien las medias de estas variables también son elevadas, estas diferencias son más pronunciadas en comparación con la mayoría de las observadas en 2004. Para otras variables en 2024, las diferencias son menores y más alineadas con lo esperado en una buena partición. Las diferencias mayores en ciertas variables podrían indicar que estas características tienen distribuciones algo distintas entre los conjuntos de entrenamiento y prueba para este año.

En conclusión, la partición de datos para el año 2004 parece haber generado conjuntos de entrenamiento y prueba con una buena homogeneidad en sus características. Para el año 2024, si bien muchas variables muestran diferencias pequeñas, algunas presentan variaciones más significativas que podrían tenerse en cuenta al interpretar el rendimiento de los modelos, aunque no necesariamente invalidan la partición realizada.

In [66]:
# Crear la variable binaria 'desocupado'
respondieron['desocupado'] = (respondieron['ESTADO'] == 2).astype(int)


In [67]:
# Separar los datos por año
respondieron_2004 = respondieron[respondieron['ANO_EPH'] == 2004].copy()
respondieron_2024 = respondieron[respondieron['ANO_EPH'] == 2024].copy()


In [68]:
# Definir las columnas a excluir
columnas_excluir = ['ESTADO', 'desocupado', 'ANO_EPH']

# Obtener las columnas disponibles en ambos años
columnas_comunes = [col for col in respondieron_2004.columns if col in respondieron_2024.columns]

# Excluir las columnas no deseadas
columnas_X = [col for col in columnas_comunes if col not in columnas_excluir]


In [69]:
# Año 2004
X_2004 = respondieron_2004[columnas_X]
y_2004 = respondieron_2004['desocupado']

# Año 2024
X_2024 = respondieron_2024[columnas_X]
y_2024 = respondieron_2024['desocupado']


In [70]:
# 1) Selecciono solo las columnas numéricas de X
columnas_num = respondieron_2004[columnas_X].select_dtypes(include='number').columns.tolist()

# 2) Redefino X_2004 y X_2024 solo con numéricas
X_2004 = respondieron_2004[columnas_num]
X_2024 = respondieron_2024[columnas_num]
y_2004 = respondieron_2004['desocupado']
y_2024 = respondieron_2024['desocupado']


In [71]:
from sklearn.model_selection import train_test_split

# Dividir 2004
X_train_2004, X_test_2004, y_train_2004, y_test_2004 = train_test_split(
    X_2004, y_2004, test_size=0.3, random_state=444)

# Dividir 2024
X_train_2024, X_test_2024, y_train_2024, y_test_2024 = train_test_split(
    X_2024, y_2024, test_size=0.3, random_state=444)


In [72]:
import numpy as np

# Función para agregar una columna de unos
def agregar_columna_unos(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

# Aplicar a los conjuntos
X_train_2004 = agregar_columna_unos(X_train_2004.values)
X_test_2004 = agregar_columna_unos(X_test_2004.values)
X_train_2024 = agregar_columna_unos(X_train_2024.values)
X_test_2024 = agregar_columna_unos(X_test_2024.values)


In [73]:
import pandas as pd

def diferencia_medias(X_train, X_test, columnas):
    df_train = pd.DataFrame(X_train[:,1:], columns=columnas)
    df_test  = pd.DataFrame(X_test[:,1:],  columns=columnas)
    medias_train = df_train.mean()
    medias_test  = df_test.mean()
    return pd.DataFrame({
        'Media Entrenamiento': medias_train,
        'Media Prueba':       medias_test,
        'Diferencia':         medias_train - medias_test
    })

# Ejemplo para 2004, tras haber filtrado/convertido:
tabla_diferencias_2004 = diferencia_medias(X_train_2004, X_test_2004, columnas_num)
print(tabla_diferencias_2004)

# Calcular para 2024
tabla_diferencias_2024 = diferencia_medias(X_train_2004, X_test_2004, columnas_num)


            Media Entrenamiento  Media Prueba  Diferencia
PP08J2                 1.153214      1.058388    0.094826
PP3E_TOT              17.232997     17.522876   -0.289879
PP03D                  0.066330      0.081481   -0.015151
PP06C                 38.494021     40.699346   -2.205325
NRO_HOGAR              1.026158      1.028322   -0.002164
PP04B2                 0.035501      0.034858    0.000642
PP08J3                 0.274664      0.466231   -0.191567
V18_M                  0.046712      0.102397   -0.055685
PP06D                 19.006166     22.428758   -3.422592
T_VI                  69.009529     61.362963    7.646566
PP08D1               193.471599    203.957298  -10.485699
PONDERA             1627.138640   1618.436601    8.702038
V8_M                   3.811659      6.017429   -2.205770
PP04B3_DIA             0.067825      0.138998   -0.071173
PP08F1                 7.383221      3.233115    4.150106
V9_M                   1.130419      1.023965    0.106453
PP11G_MES     